# CineCue: 4. Rating baselines

Run notebooks 1–3 first. Phase 2 starts by predicting star ratings with two simple
models. Every later model must beat them:

* **Global mean:** the training mean rating for every user and movie.
* **Bias model:** mean + user bias + movie bias. A user bias captures how generously
  someone rates, and a movie bias captures how well a movie is received. Both are shrunk
  toward 0 when there are few ratings.

Models are fitted on `train.csv` and tuned on `validation.csv`. The test split stays
unused until the final comparison in notebook 5.

We score predictions in two ways:

| Protocol | What a prediction may use |
| --- | --- |
| **Frozen** | Only the training split. Profiles are fixed at the training cutoff. |
| **Online** | The training split plus the same user's validation ratings at strictly earlier timestamps. Movie parameters stay fixed. |

The online protocol matches a live service, where a new user's profile grows with every
rating. It matters here because 611 users first appear after the training cutoff.
Ratings that share a timestamp never inform one another, and no rating informs itself.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Work from either the repository root or its notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "requirements.txt").is_file(), "Open this notebook from the cloned repository."
sys.path.insert(0, str(ROOT / "src"))
PROCESSED = ROOT / "data" / "processed"
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.4f}".format)

from cinecue.data import load_splits
from cinecue.evaluation import by_segment, overall, segment_labels
from cinecue.models import BiasModel, GlobalMean, online_predict, rmse

splits = load_splits(PROCESSED)
train, validation = splits["train"], splits["validation"]
for name, frame in splits.items():
    print(f"{name:>10}: {len(frame):,} ratings, {frame.user_id.nunique():,} users, {frame.movie_id.nunique():,} movies")
assert train.timestamp.max() < validation.timestamp.min()

     train: 800,168 ratings, 5,400 users, 3,662 movies
validation: 100,020 ratings, 1,134 users, 3,305 movies
      test: 100,021 ratings, 1,209 users, 3,407 movies


## Tuning the bias model

We score each pair of user and movie regularization strengths on validation RMSE.
Larger values pull the biases of sparse users and movies toward 0. We select the
pair with the lowest **online** RMSE, because that is how the model would run in
service, and report the frozen score next to it. All predictions are clipped to the
1–5 star range.

In [2]:
global_mean = GlobalMean().fit(train.user_id, train.movie_id, train.rating)
mean_prediction = global_mean.predict(validation.user_id, validation.movie_id)

grid = []
for user_reg in [2, 5, 10, 25]:
    for item_reg in [2, 5, 10, 25]:
        model = BiasModel(user_reg=user_reg, item_reg=item_reg).fit(train.user_id, train.movie_id, train.rating)
        grid.append({"user_reg": user_reg, "item_reg": item_reg,
                     "frozen RMSE": rmse(validation.rating, model.predict(validation.user_id, validation.movie_id)),
                     "online RMSE": rmse(validation.rating, online_predict(model, train, validation))})
grid = pd.DataFrame(grid)
best = grid.loc[grid["online RMSE"].idxmin()]
display(grid.pivot(index="user_reg", columns="item_reg", values="online RMSE").rename_axis(
    index="user_reg \\ item_reg"))
print(f"Selected user_reg={best.user_reg:g}, item_reg={best.item_reg:g}: "
      f"online RMSE {best['online RMSE']:.4f}, frozen RMSE {best['frozen RMSE']:.4f}")

item_reg,2,5,10,25
user_reg \ item_reg,,,,
2,0.9155,0.9157,0.9163,0.9189
5,0.9141,0.9143,0.9149,0.9174
10,0.9141,0.9142,0.9148,0.9173
25,0.9164,0.9165,0.9171,0.9196


Selected user_reg=10, item_reg=2: online RMSE 0.9141, frozen RMSE 0.9678


In [3]:
bias = BiasModel(user_reg=best.user_reg, item_reg=best.item_reg).fit(train.user_id, train.movie_id, train.rating)
predictions = {"global mean": mean_prediction,
               "bias (frozen)": bias.predict(validation.user_id, validation.movie_id),
               "bias (online)": online_predict(bias, train, validation)}
results = overall(validation.rating, predictions)
display(results)
gain = 1 - results.loc["bias (online)", "RMSE"] / results.loc["global mean", "RMSE"]
display(Markdown(f"**Interpretation:** The global mean scores RMSE {results.loc['global mean', 'RMSE']:.3f}. "
                 f"Frozen biases reduce it to {results.loc['bias (frozen)', 'RMSE']:.3f}, and updating user "
                 f"biases online reduces it to {results.loc['bias (online)', 'RMSE']:.3f}, "
                 f"{gain:.1%} below the global mean. Notebook 5 has to beat these numbers."))

,RMSE,MAE
global mean,1.1191,0.9366
bias (frozen),0.9678,0.7694
bias (online),0.9141,0.7211


**Interpretation:** The global mean scores RMSE 1.119. Frozen biases reduce it to 0.968, and updating user biases online reduces it to 0.914, 18.3% below the global mean. Notebook 5 has to beat these numbers.

## Where the errors are

The README requires reporting sparse histories and infrequent movies separately. We
label each validation rating by how many **training** ratings its user and movie have.
A count of 0 is a cold start. Fallbacks are built into the model:

| Case | Frozen prediction | Online prediction |
| --- | --- | --- |
| Known user and movie | mean + user bias + movie bias | same, with the user bias updated |
| New user | mean + movie bias | user bias learned from their earlier ratings |
| New movie | mean + user bias | same, with the user bias updated |
| Both new | mean | mean + online user bias |

In [4]:
labels = segment_labels(validation, train)
user_table = by_segment(validation.rating, predictions, labels["user history"])
movie_table = by_segment(validation.rating, predictions, labels["movie history"])
display(user_table.rename_axis("training ratings per user"))
display(movie_table.rename_axis("training ratings per movie"))
new_users = user_table.loc["0 (new)"]
display(Markdown(
    f"**Interpretation:** {new_users.rows:,.0f} of {len(validation):,} validation ratings "
    f"({new_users.rows / len(validation):.1%}) come from users with no training history. Frozen, they "
    f"receive only movie biases (RMSE {new_users['bias (frozen)']:.3f}). Online, the same model reaches "
    f"{new_users['bias (online)']:.3f} once their own ratings arrive. Movies with 0 training ratings are "
    f"rare ({movie_table.loc['0 (new)', 'rows'] if '0 (new)' in movie_table.index else 0:,.0f} rows) and stay "
    "the hardest case, because nothing is known about them."))

,rows,global mean,bias (frozen),bias (online)
training ratings per user,,,,
0 (new),76176,1.1219,0.9855,0.9217
1-19,2725,1.0426,0.9240,0.8935
20-99,5390,1.0571,0.9007,0.8805
100-299,7644,1.0581,0.8680,0.8620
300+,8085,1.2099,0.9464,0.9186


,rows,global mean,bias (frozen),bias (online)
training ratings per movie,,,,
0 (new),17,1.4647,1.3839,1.3057
1-19,875,1.4040,1.2288,1.1665
20-99,7362,1.3099,1.0394,0.9736
100-499,40292,1.1559,0.9805,0.9195
500+,51474,1.0527,0.9416,0.8958


**Interpretation:** 76,176 of 100,020 validation ratings (76.2%) come from users with no training history. Frozen, they receive only movie biases (RMSE 0.985). Online, the same model reaches 0.922 once their own ratings arrive. Movies with 0 training ratings are rare (17 rows) and stay the hardest case, because nothing is known about them.

In [5]:
summary = {"validation_rows": len(validation),
           "selected": {"user_reg": float(best.user_reg), "item_reg": float(best.item_reg)},
           "grid": grid.to_dict(orient="records"),
           "validation": results.to_dict(orient="index"),
           "by_user_history": user_table.reset_index(names="segment").to_dict(orient="records"),
           "by_movie_history": movie_table.reset_index(names="segment").to_dict(orient="records")}
(PROCESSED / "baseline_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("Saved baseline_summary.json to data/processed/.")

Saved baseline_summary.json to data/processed/.


Notebook 5 fits matrix factorization against these baselines, tests whether UTC context
helps, and runs the one test-set comparison of all rating models.